# Intrinsic Wavefront: Higher-Order Zernikes Zj (v2)

**Author:** Aaron Roodman  
**Date Created:** 2026-04-22  
**Last Modified:** 2026-04-22  
**Status:** Draft  
**Keywords:** AOS, Intrinsic Wavefront, Higher-Order Zernikes, Full Array Mode  

## Description

Extend the Z4 check (`intrinsics_checkZ4`) to higher-order Zernike coefficients Zj (j = 5 … 17 by default). For each Zj:

1. Start from the Danish v1.0 FAM mktable parquet outputs for 2026-03-15 → 2026-04-09, and restrict to donuts whose visit has rotator angle near 0°. At rotator ≈ 0° the OCS and CCS field-angle frames coincide, which avoids the rotator-dependent frame confusion that appears on non-rotationally-symmetric Zernikes such as Z5/Z6.
2. Drop any `(day_obs, seq_num)` whose count of good donuts (after the rotator + matched_intra_extra cuts) is below `MIN_DONUTS_PER_VISIT` — visits with partial focal-plane coverage otherwise distort the binned maps.
3. Subtract the per-visit linear (k1, k2, k3) fit from the data Zj using the matching `_fits.parquet` file.
4. Compute two independent intrinsic Zj maps:
   - **Pipeline** — per-donut `zk_intrinsic_OCS` from the parquet (what the AOS pipeline actually subtracts).
   - **Batoid v2** — `batoid_rubin.LSSTBuilder.build_det` + `batoid.zernike` per CCD, following the recipe from `Intrinsic Zk.ipynb`. Includes the per-CCD focal-plane height map via `LSSTBuilder.ccd_height_map_dir`.
5. Produce one 2×3 focal-plane page per Zj: Data (Zj − linear), Pipeline intrinsic, Batoid v2 intrinsic, Data−Pipeline, Data−Batoid v2, Pipeline−Batoid v2. Colour scales are grouped: the three top panels (Data + two intrinsics) share one `(vmin, vmax)` from the per-map 3 %/97 % percentiles; the two Data−Intrinsic diffs share another; the Pipeline−Batoid v2 panel gets its own.

**Output:** In-notebook plots and one multi-page PDF at `PDF_OUTPUT` (one page per Zj).

**Based on:**
- `aos/intrinsics_checkZ4.ipynb` — per-CCD Batoid v2 intrinsic and linear-fit handling
- `~/Astrophysics/Code/Claude/Intrinsic Zk.ipynb` — batoid_rubin per-CCD recipe
- `intrinsics_mktable.ipynb` / `code/intrinsics_lib.py` — streaming parquet donut/visits format
- `code/dz_fitting.py` — focal-plane Zernike basis for the per-visit linear fit

<a id='changelog'></a>
## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-22 | Aaron Roodman | Initial version — Z5 … Z17, rotator ≈ 0° slice, Pipeline vs Batoid v2 |
| 2026-04-22 | Aaron Roodman | v2: switched input from HDF5 to Danish v1.0 parquet (donuts + sidecar visits + fits). Added `MIN_DONUTS_PER_VISIT` cut that drops any (day_obs, seq_num) with fewer than 500 good donuts after the rotator + matched cuts. Colour scaling reworked: per-group 3 %/97 % percentiles with three groups — (Data, Pipeline, Batoid v2), (Data−Pipeline, Data−Batoid v2), (Pipeline−Batoid v2). |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Input donuts parquet (Danish v1.0 results). The streaming mktable writer
# now uses parquet (list<float> for the Zernike arrays) with a sidecar
# visits parquet at {stem}_visits.parquet.
INPUT_DONUTS = 'output/aos_fam_danish_v1_triplets_20260315_20260409.parquet'

# Linear-fit parquet (auto-derived as {stem}_fits.parquet if None)
FIT_PARQUET = None

# Coordinate system used for the linear fit (must match the fit parquet)
COORD_SYS = 'OCS'

# Focal-plane radius for the linear-fit basis normalization (degrees),
# matches code/dz_fitting.py: focal_plane_zernike_basis default.
FP_RADIUS_DEG = 1.75

# Zernike Noll indices to plot. One PDF page per Zj.
ZJ_LIST = list(range(5, 18))   # 5..17 inclusive

# Rotator-angle selection (degrees). Only donuts whose visit has
# rotator_angle within ROTATOR_TOL_DEG of ROTATOR_CENTER_DEG are kept.
# At rotator ≈ 0° the OCS and CCS field-angle frames are aligned, so the
# OCS-stored data can be compared directly against the Batoid v2 intrinsic
# evaluated in CCS.
ROTATOR_CENTER_DEG = 0.0
ROTATOR_TOL_DEG    = 5.0

# Per-visit quality cut: drop any (day_obs, seq_num) whose number of
# good donuts (after the rotator + matched_intra_extra cuts) is below
# MIN_DONUTS_PER_VISIT. Partial-coverage or failed visits otherwise
# distort the focal-plane maps.
MIN_DONUTS_PER_VISIT = 500

# Per-CCD intrinsic grid for Batoid v2 (same convention as the Z4 notebook).
INTRINSIC_PER_CCD_GRID_N = 8

# Batoid v2 config (from 'Intrinsic Zk.ipynb').
BATOID_V2_YAML         = 'LSST_r.yaml'
BATOID_V2_WAVELENGTH_M = 620e-9
BATOID_V2_JMAX         = 28
BATOID_V2_EPS          = 0.612
BATOID_V2_NX           = 63

# Writable data directory for batoid_rubin (FEA / bending / CCD height map
# Zenodo datasets). Setup cell sets BATOID_RUBIN_DATA_DIR only if not
# already present in the shell environment.
BATOID_RUBIN_DATA_DIR = '~/batoid_rubin_data'

# Focal-plane map binning
FP_NSTEPS = 257
FP_MAX_MM = 320.0

# Restrict to donuts whose intra and extra centroids matched
REQUIRE_MATCHED = True

# Per-group colour-scale percentiles for the results 2×3 grid.
# vmin = min over group of the low percentile; vmax = max over group of
# the high percentile.  Groups: (Data, Pipeline, Batoid v2) share one
# scale, (Data−Pipeline, Data−Batoid v2) share another, Pipeline−Batoid v2
# gets its own.
PLOT_PCTL_LOW  = 3.0
PLOT_PCTL_HIGH = 97.0

# Output PDF (one page per Zj)
PDF_OUTPUT = 'output/intrinsic_Zj.pdf'

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from mpl_toolkits import axes_grid1

from scipy.stats import binned_statistic_2d
from scipy.interpolate import LinearNDInterpolator
import tqdm

from lsst.obs.lsst import LsstCam
from lsst.afw.cameraGeom import FIELD_ANGLE

# Writable batoid_rubin data dir — the default under the installed conda env
# is read-only and the first LSSTBuilder call downloads FEA / bending /
# ccd_height_map datasets from Zenodo. Respect any user-set env var.
os.environ.setdefault(
    'BATOID_RUBIN_DATA_DIR',
    os.path.expanduser(BATOID_RUBIN_DATA_DIR),
)
Path(os.environ['BATOID_RUBIN_DATA_DIR']).mkdir(parents=True, exist_ok=True)
print(f"BATOID_RUBIN_DATA_DIR = {os.environ['BATOID_RUBIN_DATA_DIR']}")

import batoid
from batoid_rubin import LSSTBuilder

# Shared cameraGeom helpers (common/camera_utils.py)
sys.path.insert(0, str(Path.cwd().parent))
from common.camera_utils import pixel_to_focal  # noqa: E402

plt.rcParams['figure.dpi'] = 110

<a id='functions'></a>
## Helper Functions

In [ ]:
def add_colorbar(im, aspect=20, pad_fraction=0.5, **kwargs):
    """Attach a vertical colorbar matched to the host axes' height."""
    divider = axes_grid1.make_axes_locatable(im.axes)
    width = axes_grid1.axes_size.AxesY(im.axes, aspect=1.0 / aspect)
    pad = axes_grid1.axes_size.Fraction(pad_fraction, width)
    current_ax = plt.gca()
    cax = divider.append_axes('right', size=width, pad=pad)
    plt.sca(current_ax)
    return im.axes.figure.colorbar(im, cax=cax, **kwargs)


# ------------------------------------------------------------------
# Donut → focal-plane coordinates
# ------------------------------------------------------------------

def compute_fp_coords(aosTable, camera, x_col='centroid_x_intra',
                      y_col='centroid_y_intra'):
    """Evaluate fpx/fpy in mm for every donut via the per-detector
    cameraGeom PIXELS → FOCAL_PLANE transform (vectorised per sensor).
    Works with either an astropy QTable or a pandas DataFrame.
    """
    det_names = np.asarray(aosTable['detector']).astype(str)
    x_pix = np.asarray(aosTable[x_col], dtype=float)
    y_pix = np.asarray(aosTable[y_col], dtype=float)
    fpx = np.full_like(x_pix, np.nan)
    fpy = np.full_like(y_pix, np.nan)
    for det in camera:
        name = det.getName()
        mask = (det_names == name)
        if not np.any(mask):
            continue
        fx, fy = pixel_to_focal(x_pix[mask], y_pix[mask], det)
        fpx[mask] = fx
        fpy[mask] = fy
    return fpx, fpy


def _per_ccd_field_angle_grid(det, grid_n):
    """n×n CCS field-angle grid (radians) covering one CCD's footprint.

    Same DVCS→CCS transpose used in the Z4 notebook and `Intrinsic Zk.ipynb`.
    """
    corners = det.getCorners(FIELD_ANGLE)
    xmin = min(c.y for c in corners)
    xmax = max(c.y for c in corners)
    ymin = min(c.x for c in corners)
    ymax = max(c.x for c in corners)
    xs = np.linspace(xmin, xmax, grid_n + 1)
    ys = np.linspace(ymin, ymax, grid_n + 1)
    xs = 0.5 * (xs[1:] + xs[:-1])
    ys = 0.5 * (ys[1:] + ys[:-1])
    xx, yy = np.meshgrid(xs, ys)
    return xx.ravel(), yy.ravel()


# ------------------------------------------------------------------
# Batoid v2 multi-Zj intrinsic
# ------------------------------------------------------------------

def build_zk_intrinsic_per_ccd_batoid_v2(camera, zj_list,
                                         yaml_name=BATOID_V2_YAML,
                                         wavelength_m=BATOID_V2_WAVELENGTH_M,
                                         grid_n=INTRINSIC_PER_CCD_GRID_N,
                                         jmax=BATOID_V2_JMAX,
                                         eps=BATOID_V2_EPS,
                                         nx=BATOID_V2_NX,
                                         verbose=True):
    """Per-CCD Batoid v2 intrinsic for a list of Zernike indices.

    Mirrors the Z4 notebook's `build_z4_intrinsic_per_ccd_batoid_v2`, but
    evaluates ``batoid.zernike`` once per grid point (it returns all Zk up
    to ``jmax`` in one call) and builds a **vector-valued**
    ``LinearNDInterpolator`` per CCD whose values have shape
    ``(n_pts, len(zj_list))``.

    Returns
    -------
    interp_dict : dict {det_name: LinearNDInterpolator}
        Query each with (thx_deg, thy_deg); result is (n_query, n_zj) in μm.
    """
    zj_list = list(zj_list)
    if max(zj_list) > jmax:
        raise ValueError(f'jmax={jmax} too small for zj_list={zj_list}')

    fiducial = batoid.Optic.fromYaml(yaml_name)
    builder = LSSTBuilder(
        fiducial, dof_coord_system='OCS',
        flip_m2_bending_modes=False, dof_angle_units='degree',
    )
    scale_um = wavelength_m * 1.0e6   # waves × λ(m) × 1e6 = μm

    interp_dict = {}
    it = camera
    if verbose:
        print(f'batoid_rubin v2 multi-Zj: {grid_n}×{grid_n} grid, '
              f'zj_list={zj_list}, λ={wavelength_m*1e9:.1f} nm')
        it = tqdm.tqdm(list(camera), desc='batoid v2 per CCD')
    for det in it:
        telescope = builder.build_det(det.getId())
        thx_rad, thy_rad = _per_ccd_field_angle_grid(det, grid_n)
        values_um = np.empty((len(thx_rad), len(zj_list)))
        for i, (x, y) in enumerate(zip(thx_rad, thy_rad)):
            zk_all = batoid.zernike(
                telescope, theta_x=float(x), theta_y=float(y),
                wavelength=wavelength_m, projection='gnomonic',
                jmax=jmax, eps=eps, nx=nx,
            )
            for k, zj in enumerate(zj_list):
                values_um[i, k] = zk_all[zj] * scale_um
        thx_deg = np.rad2deg(thx_rad)
        thy_deg = np.rad2deg(thy_rad)
        pts = np.column_stack([thx_deg, thy_deg])
        interp_dict[det.getName()] = LinearNDInterpolator(
            pts, values_um, fill_value=np.nan,
        )
    return interp_dict


def evaluate_zk_per_ccd_multi(det_names, thx_deg, thy_deg, interp_dict, n_zj):
    """Dispatch each donut to its CCD's multi-Zj interpolator.
    Returns (n_donuts, n_zj) array in μm."""
    out = np.full((len(det_names), n_zj), np.nan, dtype=float)
    for name, interp in interp_dict.items():
        mask = (det_names == name)
        if not np.any(mask):
            continue
        out[mask] = interp(thx_deg[mask], thy_deg[mask])
    return out


# ------------------------------------------------------------------
# Linear (k1, k2, k3) basis evaluation (same basis as dz_fitting.py)
# ------------------------------------------------------------------

def linear_fit_values(thx_deg, thy_deg, k1, k2, k3, fp_radius=FP_RADIUS_DEG):
    """k1 + 2·k2·(thx_deg/R) + 2·k3·(thy_deg/R) in μm.
    Matches `code/dz_fitting.focal_plane_zernike_basis` (Z1 piston, Z2 tilt,
    Z3 tip on the unit disk).
    """
    x = thx_deg / fp_radius
    y = thy_deg / fp_radius
    return k1 + 2.0 * k2 * x + 2.0 * k3 * y


# ------------------------------------------------------------------
# Binning + plotting
# ------------------------------------------------------------------

def bin_median_2d(xval, yval, zval, xbins, ybins):
    """Median of zval on an (xbins × ybins) 2D grid over (xval, yval)."""
    stat, x_edges, y_edges, _ = binned_statistic_2d(
        xval, yval, zval, statistic='median', bins=[xbins, ybins]
    )
    return stat, x_edges, y_edges


def _group_limits(stats, q_lo=PLOT_PCTL_LOW, q_hi=PLOT_PCTL_HIGH):
    """Per-group (vmin, vmax): for each map in the group take its (q_lo, q_hi)
    percentiles; combine as vmin = min across maps of the low percentile,
    vmax = max across maps of the high percentile. Outer envelope, asymmetric.
    """
    lows, highs = [], []
    for s in stats:
        flat = s.ravel()
        flat = flat[np.isfinite(flat)]
        if flat.size == 0:
            continue
        lo, hi = np.percentile(flat, [q_lo, q_hi])
        lows.append(lo)
        highs.append(hi)
    if not lows or not highs:
        return -1.0, 1.0
    return float(min(lows)), float(max(highs))


def plot_fp_grid(maps, xe, ye, title, cmap='RdBu_r',
                 groups=((0, 1, 2), (3, 4), (5,)),
                 q_lo=PLOT_PCTL_LOW, q_hi=PLOT_PCTL_HIGH):
    """Render six focal-plane maps on a 2×3 grid with grouped colour scales.

    `groups` partitions the six panels into colour-scale groups (defaults
    match the Data / Data−Intrinsic / Intrinsic−Intrinsic layout used here).
    Each group's (vmin, vmax) comes from per-map (q_lo, q_hi) percentiles
    via `_group_limits`; all panels use the same diverging cmap.
    """
    assert len(maps) == 6
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))

    # Assign (vmin, vmax) per panel from its group
    idx_lim = [None] * 6
    for group in groups:
        stats = [maps[gi][1] for gi in group]
        vmin, vmax = _group_limits(stats, q_lo=q_lo, q_hi=q_hi)
        for gi in group:
            idx_lim[gi] = (vmin, vmax)

    for idx, (subtitle, stat) in enumerate(maps):
        ax = axes.flat[idx]
        vmin, vmax = idx_lim[idx]
        im = ax.imshow(
            stat.T, origin='lower',
            extent=[xe[0], xe[-1], ye[0], ye[-1]],
            cmap=cmap, interpolation='none', aspect='equal',
            vmin=vmin, vmax=vmax,
        )
        ax.set_xlabel('fpx [mm]')
        ax.set_ylabel('fpy [mm]')
        ax.set_title(subtitle)
        add_colorbar(im).set_label('Z [μm]')
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    return fig

<a id='data'></a>
## Data Access

In [ ]:
# Danish v1.0 parquet inputs:
#   donuts parquet      = INPUT_DONUTS
#   visits parquet      = {stem}_visits.parquet   (sidecar written by mktable)
#   fits  parquet       = {stem}_fits.parquet     (or FIT_PARQUET if set)
donuts_path = Path(INPUT_DONUTS)
if not donuts_path.exists():
    raise FileNotFoundError(f'Donuts parquet not found: {donuts_path}')

visits_path = donuts_path.parent / f'{donuts_path.stem}_visits.parquet'
if not visits_path.exists():
    raise FileNotFoundError(f'Visits parquet not found: {visits_path}')

if FIT_PARQUET is None:
    fits_path = donuts_path.parent / f'{donuts_path.stem}_fits.parquet'
else:
    fits_path = Path(FIT_PARQUET)
if not fits_path.exists():
    raise FileNotFoundError(f'Fit parquet not found: {fits_path}')

print(f'Donuts:      {donuts_path}')
print(f'Visits:      {visits_path}')
print(f'Fit parquet: {fits_path}')

aosTable   = pd.read_parquet(donuts_path)
visit_info = pd.read_parquet(visits_path)
fits_table = pd.read_parquet(fits_path)
print(f'Loaded {len(aosTable):,} donuts, {len(aosTable.columns)} columns, '
      f'{len(visit_info)} visits')
print(f'Fit rows: {len(fits_table)}, columns: {len(fits_table.columns)}')

In [ ]:
# Build the per-donut keep mask in three stages:
#   1) Rotator window: |rotator_angle − ROTATOR_CENTER_DEG| ≤ ROTATOR_TOL_DEG
#   2) Matched intra/extra (optional)
#   3) Per-visit donut count ≥ MIN_DONUTS_PER_VISIT after (1)+(2)

# 1) Rotator. The Danish v1.0 parquet keeps rotator_angle in the visits
#    sidecar rather than duplicated per donut, so we look it up via a
#    (day_obs, seq_num) merge. Fall back to a per-donut column if present.
if 'rotator_angle' in aosTable.columns:
    rot = np.asarray(aosTable['rotator_angle'], dtype=float)
    rot_source = 'donuts'
else:
    if 'rotator_angle' not in visit_info.columns:
        raise KeyError("Neither the donuts parquet nor the visits parquet "
                       "carries a 'rotator_angle' column.")
    rot_lookup = visit_info[['day_obs', 'seq_num', 'rotator_angle']]
    donuts_key = aosTable[['day_obs', 'seq_num']]
    merged_rot = donuts_key.merge(rot_lookup, on=['day_obs', 'seq_num'],
                                  how='left')
    rot = merged_rot['rotator_angle'].to_numpy(dtype=float)
    rot_source = 'visits'
print(f"rotator_angle source: {rot_source} parquet")

# Sanity-check units and auto-convert radians → degrees if the range is tiny.
if np.all(np.abs(rot[np.isfinite(rot)]) <= 2 * np.pi + 0.1):
    print('rotator_angle appears to be in radians; converting to degrees.')
    rot = np.rad2deg(rot)
rmin, rmax = float(np.nanmin(rot)), float(np.nanmax(rot))
print(f'rotator_angle range [deg]: [{rmin:+.2f}, {rmax:+.2f}]')

rot_mask = np.abs(rot - ROTATOR_CENTER_DEG) <= ROTATOR_TOL_DEG
# Treat NaN rotator as a fail.
rot_mask &= np.isfinite(rot)
print(f'Rotator cut |rot − {ROTATOR_CENTER_DEG:.1f}°| ≤ {ROTATOR_TOL_DEG:.1f}°: '
      f'{int(rot_mask.sum()):,}/{len(rot_mask):,} donuts')

# 2) Matched intra/extra.
keep = rot_mask.copy()
if REQUIRE_MATCHED and 'matched_intra_extra' in aosTable.columns:
    keep &= np.asarray(aosTable['matched_intra_extra'], dtype=bool)
print(f'After matched_intra_extra cut: {int(keep.sum()):,} donuts kept')

# 3) Per-visit good-donut count threshold.
day_obs_arr = np.asarray(aosTable['day_obs'], dtype=np.int64)
seq_num_arr = np.asarray(aosTable['seq_num'], dtype=np.int64)
_visit_id = day_obs_arr.astype(np.int64) * 100000 + seq_num_arr.astype(np.int64)
uniq_visits, counts = np.unique(_visit_id[keep], return_counts=True)
good_visits = set(uniq_visits[counts >= MIN_DONUTS_PER_VISIT].tolist())
n_total_visits_seen = len(uniq_visits)
n_good_visits = len(good_visits)
print(f'Visits with ≥ {MIN_DONUTS_PER_VISIT} good donuts: '
      f'{n_good_visits}/{n_total_visits_seen}')
visit_ok = np.fromiter((vid in good_visits for vid in _visit_id),
                       count=len(_visit_id), dtype=bool)
keep &= visit_ok
print(f'After per-visit count cut: {int(keep.sum()):,} donuts kept '
      f'across {n_good_visits} visits')

<a id='analysis'></a>
## Analysis

In [ ]:
# Noll index bookkeeping for the stored zk columns.
zk_col      = f'zk_{COORD_SYS}'
zk_intr_col = f'zk_intrinsic_{COORD_SYS}'
thx_col     = f'thx_{COORD_SYS}'
thy_col     = f'thy_{COORD_SYS}'

# zk_OCS / zk_intrinsic_OCS come from parquet list<float> columns, one
# array per donut. np.stack(...tolist()) gives a (N, n_Z) float array.
zk_data_full = np.stack(aosTable[zk_col].to_list())          # (N, n_Z), μm
zk_intr_full = np.stack(aosTable[zk_intr_col].to_list())     # (N, n_Z), μm
nZk = zk_data_full.shape[1]

if 'nollIndices' in visit_info.columns:
    iZs_all = [int(n) for n in visit_info['nollIndices'].iloc[0]]
    if len(iZs_all) != nZk:
        iZs_all = list(range(4, 4 + nZk))
else:
    iZs_all = list(range(4, 4 + nZk))
iZidx_all = {iZ: i for i, iZ in enumerate(iZs_all)}
print(f'Zernike Noll indices in parquet: {iZs_all}')

missing = [j for j in ZJ_LIST if j not in iZidx_all]
if missing:
    raise KeyError(f'Requested Zj not present in data: {missing}')
print(f'Requested Zj to plot: {ZJ_LIST}')

# Field angles (deg) in the fit coordinate system.
thx_deg = np.rad2deg(np.asarray(aosTable[thx_col], dtype=float))
thy_deg = np.rad2deg(np.asarray(aosTable[thy_col], dtype=float))

# Focal-plane (fpx, fpy) in mm from the intra-focal pixel centroid via
# per-detector cameraGeom PIXELS → FOCAL_PLANE transforms.
camera = LsstCam.getCamera()
fpx, fpy = compute_fp_coords(aosTable, camera,
                             x_col='centroid_x_intra',
                             y_col='centroid_y_intra')
print(f'fpx range [mm]: [{np.nanmin(fpx):+.1f}, {np.nanmax(fpx):+.1f}]; '
      f'fpy range [mm]: [{np.nanmin(fpy):+.1f}, {np.nanmax(fpy):+.1f}]')

det_names_all = np.asarray(aosTable['detector']).astype(str)

In [ ]:
# Subtract the per-visit linear (k1, k2, k3) fit from the data Zj. The fits
# parquet carries z1toz3_z{j}_c{1,2,3} columns for each Zj in the iZs list.
day_obs = np.asarray(aosTable['day_obs'], dtype=int)
seq_num = np.asarray(aosTable['seq_num'], dtype=int)
donuts_df = pd.DataFrame({'day_obs': day_obs, 'seq_num': seq_num})

Zj_data_linear_um = {}   # {j: ndarray of shape (N,)}  data − linear fit (μm)
for j in ZJ_LIST:
    c1, c2, c3 = f'z1toz3_z{j}_c1', f'z1toz3_z{j}_c2', f'z1toz3_z{j}_c3'
    missing = [c for c in (c1, c2, c3) if c not in fits_table.columns]
    if missing:
        raise KeyError(f'Missing fit columns for Z{j}: {missing}')
    keys_j = fits_table[['day_obs', 'seq_num', c1, c2, c3]].copy()
    merged = donuts_df.merge(keys_j, on=['day_obs', 'seq_num'], how='left')
    k1 = merged[c1].to_numpy()
    k2 = merged[c2].to_numpy()
    k3 = merged[c3].to_numpy()
    Zj_linear = linear_fit_values(thx_deg, thy_deg, k1, k2, k3,
                                   fp_radius=FP_RADIUS_DEG)
    col_idx = iZidx_all[j]
    Zj_data_linear_um[j] = zk_data_full[:, col_idx] - Zj_linear

# Pipeline intrinsic: direct slice of zk_intrinsic_{COORD_SYS}.
Zj_pipe_um = {j: zk_intr_full[:, iZidx_all[j]] for j in ZJ_LIST}

print('Per-donut statistics after linear correction (kept donuts, μm):')
print(f"  {'Zj':>4s}  {'mean_data':>11s}  {'std_data':>10s}  "
      f"{'mean_pipe':>11s}  {'std_pipe':>10s}")
for j in ZJ_LIST:
    ad = Zj_data_linear_um[j][keep]
    ap = Zj_pipe_um[j][keep]
    print(f'  Z{j:<3d}  {np.nanmean(ad):+11.4f}  {np.nanstd(ad):10.4f}  '
          f'{np.nanmean(ap):+11.4f}  {np.nanstd(ap):10.4f}')

In [ ]:
# Batoid v2 per-CCD multi-Zj intrinsic. One LSSTBuilder and one
# LinearNDInterpolator per detector; each interpolator returns
# (n_query, len(ZJ_LIST)) in μm.
interp_batoidv2 = build_zk_intrinsic_per_ccd_batoid_v2(
    camera, ZJ_LIST, grid_n=INTRINSIC_PER_CCD_GRID_N,
)
Zj_bv2_matrix = evaluate_zk_per_ccd_multi(
    det_names_all, thx_deg, thy_deg, interp_batoidv2, len(ZJ_LIST),
)   # (N, len(ZJ_LIST))
Zj_bv2_um = {j: Zj_bv2_matrix[:, k] for k, j in enumerate(ZJ_LIST)}

print('Batoid v2 intrinsic per-donut statistics (kept donuts, μm):')
for j in ZJ_LIST:
    a = Zj_bv2_um[j][keep]
    print(f'  Z{j:<3d}  mean={np.nanmean(a):+11.4f}  std={np.nanstd(a):10.4f}')

<a id='results'></a>
## Results & Plots

In [ ]:
# Common binning and PDF figure collector. Re-running this cell starts a
# fresh collection. `_collect` returns None so cells don't render a second
# inline copy of the figure.
xbins = np.linspace(-FP_MAX_MM, FP_MAX_MM, FP_NSTEPS)
ybins = np.linspace(-FP_MAX_MM, FP_MAX_MM, FP_NSTEPS)

xv = fpx[keep]
yv = fpy[keep]

_pdf_figs = []


def _collect(fig):
    _pdf_figs.append(fig)

In [ ]:
# One 2×3 page per Zj:
#   Top row:    data (Z − linear),  Pipeline intrinsic,  Batoid v2 intrinsic
#   Bottom row: data − Pipeline,    data − Batoid v2,    Pipeline − Batoid v2
# Top row and bottom row each share a colour scale set from the 98th
# percentile of their three panels; all panels use RdBu_r.
n_kept = int(keep.sum())
xe = None
for j in ZJ_LIST:
    data_j = Zj_data_linear_um[j]
    pipe_j = Zj_pipe_um[j]
    bv2_j  = Zj_bv2_um[j]

    stat_data, xe, ye = bin_median_2d(xv, yv, data_j[keep], xbins, ybins)
    stat_pipe, _, _   = bin_median_2d(xv, yv, pipe_j[keep], xbins, ybins)
    stat_bv2,  _, _   = bin_median_2d(xv, yv, bv2_j[keep],  xbins, ybins)

    stat_data_pipe = stat_data - stat_pipe
    stat_data_bv2  = stat_data - stat_bv2
    stat_pipe_bv2  = stat_pipe - stat_bv2

    maps = [
        (f'Data:  Z{j} − linear',          stat_data),
        (f'Pipeline intrinsic  Z{j}',      stat_pipe),
        (f'Batoid v2 intrinsic  Z{j}',     stat_bv2),
        (f'Data  −  Pipeline  (Z{j})',     stat_data_pipe),
        (f'Data  −  Batoid v2  (Z{j})',    stat_data_bv2),
        (f'Pipeline  −  Batoid v2  (Z{j})', stat_pipe_bv2),
    ]
    fig = plot_fp_grid(
        maps, xe, ye,
        title=(f'Z{j}   |   rotator ≈ {ROTATOR_CENTER_DEG:.0f}° '
               f'(±{ROTATOR_TOL_DEG:.0f}°),   {n_kept:,} donuts'),
    )
    _collect(fig)

In [ ]:
# Write every collected figure (one per Zj) to a single PDF.
pdf_path = Path(PDF_OUTPUT)
pdf_path.parent.mkdir(parents=True, exist_ok=True)
with PdfPages(pdf_path) as pdf:
    for f in _pdf_figs:
        pdf.savefig(f, bbox_inches='tight')
print(f'Saved {len(_pdf_figs)} pages to {pdf_path}')